<a href="https://colab.research.google.com/github/Varshini0051/Rag_chatbot/blob/new/Session_3_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 0: Install the tools we need
# Think of this like downloading apps on your phone before you can use them.
# Just run this cell and wait — it'll take a minute or two!

%pip -q install langchain langchain-community langchain-text-splitters langchain-huggingface pypdf sentence-transformers faiss-cpu ipywidgets requests==2.32.4 groq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip filename.zip

In [ ]:
import logging
from transformers.utils import logging as t_logging
t_logging.set_verbosity_error()
logging.getLogger("pypdf").setLevel(logging.ERROR)

### What is RAG?

**RAG** stands for **Retrieval-Augmented Generation**. Let's break that down into plain English:

> **RAG = Search first, then answer.**

Instead of the AI answering from memory, a RAG system **looks up the answer in your document first**, and *then* writes a response.

**Think of it like an open-book exam:**
- In a closed-book exam, you rely on memory (and might get things wrong)
- In an open-book exam, you can check the textbook before writing your answer

RAG gives the AI an **open-book exam**! Here's how:

| Step | What Happens | Everyday Example |
|------|-------------|--------|
| **1. Search** (Retrieval) | The system searches your document and finds the most relevant paragraphs | Like using Ctrl+F, but smarter — it searches by *meaning*, not just exact words |
| **2. Attach** (Augmentation) | Those paragraphs are attached to your question and given to the AI | Like handing someone a cheat sheet right before asking them a question |
| **3. Answer** (Generation) | The AI reads both your question AND the paragraphs, then writes a clear answer | Like a student writing an answer after reading the relevant chapter |

<br>
<br>

<img src="https://cdn.shopaccino.com/igmguru/images/how-retrieval-augmented-generation-rag-works-5521715039778733.jpg" width="600">



```
# This is formatted as code
```

### But Wait — How Does the Computer Actually Search a Document?

We need a **smarter search** that understands meaning. But computers can't read text the way we do. So before we can search, we need to do two things:

1. **Chunking** — Break the document into small pieces (like cutting a book into paragraphs)
2. **Embeddings** — Convert each piece into numbers that capture its meaning (so the computer can compare meanings)

Together, these let us do **"search by meaning"** — which is way more powerful than Ctrl+F!

**Let's build this step by step...**

### Step 1: Reading the Document

Before we can search a document, we must first **read it into Python**.

In the real world, data can come from many sources — PDFs, text files, web pages, databases, etc. Each type needs different code to read it.

To make our life easier, we use a library called **LangChain**. It gives us ready-made tools to load, process, and work with documents.

Let's start by uploading our PDF!

In [ ]:
# Now we create a "loader" that knows how to read PDFs
# PyPDFLoader is a tool from LangChain that reads PDF files page by page

from langchain_community.document_loaders import PyPDFLoader

file_path = "HTML - Book.pdf"
loader = PyPDFLoader(file_path)

In [ ]:
# Actually read the PDF — this loads all pages into memory
# Each page becomes one "document" in our list
document = loader.load()

print("Number of pages in the document:", len(document))

In [9]:
# Let's peek at the first page to see what the loader extracted
# You'll see the text content + some metadata (info about the file)

print("First page of the document:", document[0])

First page of the document: page_content='HTML 
1 
 
HTML stands for Hypertext Markup Language, and it is the most widely used language to 
write Web Pages. 
 Hypertext refers to the way in which Web pages (HTML documents) are linked 
together. Thus, the link available on a webpage is called Hypertext. 
 
 As its name suggests, HTML is a Markup Language which means you use HTML 
to simply "mark-up" a text document with tags that tell a Web browser how to 
structure it to display. 
Originally, HTML was developed with the intent of defining the structure of documents like 
headings, paragraphs, lists, and so forth to facilitate the sharing of scientific information 
between researchers. 
Now, HTML is being widely used to format web pages with the help of different tags 
available in HTML language. 
Basic HTML Document 
In its simplest form, following is an example of an HTML document: 
<!DOCTYPE html> 
<html> 
<head> 
<title>This is document title</title> 
</head> 
<body> 
<h1>This is 

### Step 2: Chunking — Breaking the Document into Small Pieces

Our PDF might be several pages long, but AI works better with **smaller pieces** of text rather than one giant block.

**Why do we need to split it?**
- Smaller pieces = more **precise** search results
- One giant block of text = harder to find the right answer
- LLMs have a **limit** on how much text they can read at once

**Analogy:** Imagine searching through an entire textbook vs. searching through individual paragraphs. Which would be faster and more accurate? The paragraphs, right?

That's exactly what chunking does — it cuts the document into smaller, meaningful pieces.

<img src="https://substackcdn.com/image/fetch/$s_!NtgT!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Fe8febecd-ee68-42ff-ab06-41a0a3a43cd3_1102x306.gif" width="600">

In [10]:
# We use a "text splitter" to cut the document into chunks
# chunk_size=500 means each piece will be ~500 characters long (about 1 paragraph)
# chunk_overlap=20 means pieces slightly overlap so we don't lose info at the edges

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
texts = text_splitter.split_documents(document)

In [11]:
# How many chunks did we get?
# A 5-page PDF might give us around 10-15 chunks

print("Total number of chunks:", len(texts))

Total number of chunks: 27


In [12]:
# Let's look at one chunk to see what it contains
# Each chunk is a small piece of text from our PDF

print("Example chunk:")
print(texts[5].page_content)

Example chunk:
like <h1>, <div>, <p> etc. 
<h1> This tag represents the heading. 
<p> This tag represents a paragraph.


### Step 3: Turning Text into Numbers (Embeddings)

Here's a key insight: **computers don't understand words — they only understand numbers.**

So how do we make a computer understand that *"stolen painting"* and *"missing artwork"* mean similar things?

We convert each chunk of text into a **list of numbers** called an **embedding**. These numbers capture the **meaning** of the text.

**How it works:**
- Words/sentences with **similar meaning** get **similar numbers**
- Words/sentences with **different meaning** get **different numbers**

**Analogy:** Think of it like giving each chunk a GPS coordinate. Chunks about similar topics will have coordinates that are **close together** on a map, and unrelated chunks will be **far apart**.

<img src="https://miro.medium.com/v2/resize:fit:1400/0*HeeGoJ5e5myv1tsX.png" width="600">

In [13]:
# We'll use a free, pre-trained model to create our embeddings
# This model was trained on millions of sentences so it understands meaning well

from langchain_huggingface import HuggingFaceEmbeddings

In [14]:
# Load the embedding model — this might take a minute the first time
# "all-MiniLM-L6-v2" is a small but powerful model for understanding text meaning

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings model ready!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings model ready!


### Step 4: Storing the Embeddings (Vector Database)

Now that we can turn text into numbers, we need a place to **store** them so we can search through them quickly.

We use something called a **vector database** — think of it as a smart filing cabinet that:
- Keeps chunks with **similar meaning close together**
- Keeps chunks with **different meaning far apart**

This way, when you ask a question, the database can instantly find the chunks that are most related to your question!

<img src="https://weaviate.io/assets/images/vector-indexing-5f775e170bd2e2f39fa8f14153da6fdf.jpg" width="400">

In [15]:
# FAISS is a fast vector database created by Facebook/Meta
# It takes our text chunks + embedding model, converts everything to numbers,
# and organizes them for fast searching

from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(texts, embedding_model)

print("Vector store ready! Total items stored:", vector_store.index.ntotal)

Vector store ready! Total items stored: 27


### Step 5: Finding the Right Information (Retrieval)

Now comes the exciting part! We have:
- Our document split into chunks
- Each chunk converted to numbers and stored in a database

When you ask a question, the system will:
1. Convert your **question** into numbers (using the same embedding model)
2. Compare it against all the stored chunks
3. Return the **top matches** — the chunks whose meaning is closest to your question

It's like asking a librarian: *"Find me the 3 most relevant paragraphs for this question."*

In [16]:
# Create a "retriever" that will find the top 3 most relevant chunks for any question
# k=3 means: "give me the 3 best matches"

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [17]:
# Let's test it! Ask a question and see which chunks the system finds

question = "How to place the text in center?"

docs = retriever.get_relevant_documents(question)

# Print the top 3 chunks that matched our question
for i, doc in enumerate(docs, start=1):
    print(f"--- Chunk {i} ---")
    print(doc.page_content[:400])
    print()

--- Chunk 1 ---
HTML 
6 
 
<p>Hello<br /> 
You delivered your assignment on time.<br /> 
Thanks<br /> 
Mahnaz</p> 
</body> 
</html> 
This will produce the following result: 
Hello 
You delivered your assignment on time. 
Thanks 
Mahnaz 
Centering Content 
You can use <center> tag to put any content in the center of the page or any table cell. 
Example 
<!DOCTYPE html> 
<html> 
<head> 
<title>Centring Content Exam

--- Chunk 2 ---
<center> 
<p>This text is in the center.</p> 
</center> 
</body> 
</html> 
This will produce the following result: 
This text is not in the center. 
This text is in the center. 
Horizontal Lines 
Horizontal lines are used to visually break-up sections of a document. The <hr> tag 
creates a line from the current position in the document to the right margin and breaks 
the line accordingly.

--- Chunk 3 ---
HTML 
7 
 
For example, you may want to give a line between two paragraphs as in the given example 
below: 
Example 
<!DOCTYPE html> 
<html> 
<head> 
<title>

/tmp/ipython-input-827/3411934138.py:5: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retriever.get_relevant_documents(question)


### Step 6: Generating the Answer

So far we've:
- Loaded our PDF
- Split it into chunks
- Converted chunks to numbers (embeddings)
- Stored them in a vector database
- Retrieved the most relevant chunks for a question

But the retrieved chunks are just **raw text** — they're not a nice, clear answer.

This is where the **LLM** comes in! We give it:
1. Your **question**
2. The **relevant chunks** we found

And the LLM writes a **clear, human-friendly answer** based on that information. That's the "Generation" part of RAG!

In this workshop, we'll use a free AI service called **Groq** to generate answers.

In [18]:
# Read API keys from file

import random

with open("api_keys.txt", "r") as f:
    api_keys = f.read().splitlines()

random.shuffle(api_keys)

api_key = api_keys[0]

In [19]:
# Set up Groq — a free cloud AI service that will generate answers for us
# Get your own free API key at: https://console.groq.com

from groq import Groq

GROQ_API_KEY = api_key
client = Groq(api_key=GROQ_API_KEY)

def build_prompt(context, question):
    """Create the instructions we send to the AI along with the context and question."""
    return (
        f"""
            You are a document-based Question Answering assistant helping students prepare for exams.

            IMPORTANT:
            The provided document context is the ONLY source of truth.
            Answer strictly using information available in the document.
            Do NOT use outside knowledge, assumptions, or prior training information.

            Instructions:
            1. Carefully read the entire document context before answering.
            2. Extract the answer only from the provided context.
            3. If relevant information appears in multiple places, combine them logically.
            4. Do not invent, assume, or expand beyond the document.
            5. If the answer is not clearly present in the context, respond exactly with:
              Not found in document
            6. Keep answers clear, simple, and concise (maximum 2–3 sentences).

            Document Context:
            {context}

            Question:
            {question}

            Answer (based only on the document):
"""
    )

def generate_answer(prompt):
    """Send our prompt to Groq and get an answer back."""
    try:
        chat = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        return chat.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error: {e}")
        return "Not found in document."

print("Groq AI is ready!")

Groq AI is ready!


### Let's Chat With Our PDF!

Everything is set up! Our RAG pipeline is complete:
- PDF loaded and split into chunks
- Chunks converted to embeddings and stored in a vector database
- Retriever ready to find relevant chunks
- Groq AI ready to generate answers

First, let's run a **quick automated test** with 3 questions to make sure everything works. Then you can **chat freely** using the interactive chatbot below!

In [20]:
# Now try it yourself! Type any question about your PDF below.

import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import output

output.enable_custom_widget_manager()

title = widgets.HTML("<h3>PDF Chatbot</h3><p style='color:gray'>Ask anything about the uploaded document</p>")
input_box = widgets.Text(
    placeholder="Type your question here...",
    layout=widgets.Layout(width="70%")
)
ask_button = widgets.Button(
    description="Ask",
    button_style="primary",
    layout=widgets.Layout(width="100px")
)
clear_button = widgets.Button(
    description="Clear Chat",
    button_style="warning",
    layout=widgets.Layout(width="100px")
)
chat_out = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px",
        height="350px",
        overflow_y="auto"
    )
)
status = widgets.HTML("")

chat_history = []

def handle_ask(_):
    question = input_box.value.strip()
    if not question:
        return
    input_box.value = ""
    status.value = "<i style='color:gray'>Thinking...</i>"

    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    prompt = build_prompt(context, question)
    answer = generate_answer(prompt)

    chat_history.append(("You", question))
    chat_history.append(("Bot", answer))
    status.value = ""

    with chat_out:
        clear_output()
        for role, text in chat_history:
            if role == "You":
                print(f"You: {text}\n")
            else:
                print(f"Bot: {text}\n")
                print("-" * 50)

def handle_clear(_):
    chat_history.clear()
    with chat_out:
        clear_output()

ask_button.on_click(handle_ask)
clear_button.on_click(handle_clear)
input_box.on_submit(handle_ask)

buttons = widgets.HBox([ask_button, clear_button])
display(title, widgets.HBox([input_box]), buttons, status, chat_out)

HTML(value="<h3>PDF Chatbot</h3><p style='color:gray'>Ask anything about the uploaded document</p>")

HTML(value='')

Output(layout=Layout(border='1px solid #ddd', height='350px', overflow_y='auto', padding='10px'))

In [21]:
!zip resources.zip "HTML - Book.pdf" HTML_Mock_Previous_Year_Question_Paper.txt api_keys.txt

  adding: HTML - Book.pdf (deflated 7%)
  adding: HTML_Mock_Previous_Year_Question_Paper.txt (deflated 56%)
  adding: api_keys.txt (stored 0%)
